# Анализ ошибок и ограничений

Разбор случаев, когда целевой аромат (target_perfume_id) не попадает в Top-K, и ограничения модели.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "backend"))

from app.core.config import get_settings
from app.ranking.data import DataLoader
from app.ranking.profile.build_profile import session_to_user_vector
from app.ranking.scoring.score import build_sku_vectors, score_skus
import pandas as pd

In [ ]:
settings = get_settings()
loader = DataLoader(settings.data_perfume_dir, settings.data_organ_dir if settings.data_organ_dir.exists() else None)
if not loader.has_organ_data():
    print("Organ data not found; put organ_*.parquet in data/ or set DATA_ORGAN_DIR")
else:
    sessions = loader.load_organ_sessions()
    perfume_notes = loader.load_perfume_notes()
    perfumes = loader.load_perfumes()
    pv, note_to_idx, idx_to_note = build_sku_vectors(perfume_notes)
    test_sessions = sessions.sample(frac=0.2, random_state=42)
    misses = []  # session_id, target_id, rank (0 = not in top_n)
    for _, row in test_sessions.iterrows():
        sid, target = int(row["session_id"]), int(row["target_perfume_id"])
        u = session_to_user_vector(sid, loader, use_presses=True, alpha_recipe=0.7)
        if not u:
            misses.append((sid, target, -1))
            continue
        recs = score_skus(u, pv, note_to_idx, idx_to_note, top_n=50)
        ids = [r[0] for r in recs]
        rank = ids.index(target) + 1 if target in ids else 0
        if rank == 0 or rank > 10:
            misses.append((sid, target, rank))
    print("Sessions where target not in Top-10 or missing:", len(misses))
    if misses:
        print("Example (session_id, target_id, rank or 0):", misses[:5])

## Выводы по ограничениям

- **Смесь vs одна нота:** скоринг по cosine(user_vec, sku_vec) учитывает всю смесь; при малом overlap нот target может не попасть в топ.
- **Качество маппинга орган→ноты:** при неточном organ_aroma_notes_map профиль искажается.
- **Синтетика:** target в Lite задан искусственно; в реальных данных целевого выбора может не быть.
- **Рекомендация:** для улучшения — учитывать organ_feedback (view/purchased), reranking, увеличение top_n при оценке.